In [6]:
import os
from datetime import datetime

def rename_images_to_citra(folder_path):
    
    for file in os.listdir(folder_path):
        
        if file.endswith(".jpg"):
            
            try:
                name = file.replace(".jpg","")
                
                # parse original format
                dt = datetime.strptime(name, "%Y-%m-%d_%H-%M")
                
                # convert to citra format with seconds = 01
                new_name = f"citra_{dt.strftime('%Y-%m-%d_%H%M')}01.jpg"
                
                old_path = os.path.join(folder_path, file)
                new_path = os.path.join(folder_path, new_name)
                
                os.rename(old_path, new_path)
                
            except Exception as e:
                print("Skipping:", file, e)


folder = r"F:\strwaberry images\Front_row_with_text"

rename_images_to_citra(folder)

print("Renaming completed")

Renaming completed


In [8]:
import os
from datetime import datetime

def get_min_max_timestamp(folder_path):
    timestamps = []

    for file in os.listdir(folder_path):
        if file.endswith(".jpg"):
            try:
                # remove prefix and extension
                name = file.replace("citra_", "").replace(".jpg", "")
                
                # convert to datetime
                dt = datetime.strptime(name, "%Y-%m-%d_%H%M%S")
                timestamps.append(dt)

            except Exception as e:
                print(f"Skipping file {file}: {e}")

    if not timestamps:
        return None, None

    return min(timestamps), max(timestamps)


folder = r"F:\strwaberry images\Front_row_with_text\Cropped_Images_down\down_1\preprocessed_images"

min_time, max_time = get_min_max_timestamp(folder)

print("Earliest Timestamp:", min_time)
print("Latest Timestamp:", max_time)

Earliest Timestamp: 2025-12-12 15:00:01
Latest Timestamp: 2025-12-21 11:15:01


In [ ]:
import os
import pandas as pd
from datetime import datetime

def create_timestamp_excel(folder_path, output_excel):
    records = []

    for file in os.listdir(folder_path):
        if file.endswith(".jpg"):
            try:
                name = file.replace(".jpg", "")
                dt = datetime.strptime(name, "%Y-%m-%d_%H-%M")

                records.append({
                    "filename": file,
                    "date": dt.date(),
                    "time": dt.time(),
                    "timestamp": dt
                })

            except Exception as e:
                print(f"Skipping {file}: {e}")

    df = pd.DataFrame(records)

    # sort by time
    df = df.sort_values("timestamp")

    # save excel
    df.to_excel(output_excel, index=False)

    print("Excel saved to:", output_excel)


folder = r"F:\strwaberry images\Front_row_with_text\Cropped_Images_down\down_2\preprocessed_images"
output = r"F:\timestamps.xlsx"

create_timestamp_excel(folder, output)

In [9]:
import pandas as pd

def filter_datetime_excel(input_excel, output_excel, start_time, end_time):

    # read excel
    df = pd.read_excel(input_excel)

    # convert Datetime column to datetime format
    df["Date Time"] = pd.to_datetime(df["Date Time"], format="%d-%m-%Y %H:%M")

    # define time range
    start = pd.to_datetime(start_time)
    end = pd.to_datetime(end_time)

    # filter rows
    filtered_df = df[(df["Date Time"] >= start) & (df["Date Time"] <= end)]

    # save new excel
    filtered_df.to_excel(output_excel, index=False)

    print("Filtered Excel saved at:", output_excel)


input_excel = r"FAWN_2025.xlsx"
output_excel = r"Front_down_1_plants_weather_mapped.xlsx"

# start_time = "2025-12-12 16:30:01"
# end_time = "2026-01-18 18:45:01"

filter_datetime_excel(input_excel, output_excel, start_time=min_time, end_time=max_time)

Filtered Excel saved at: Front_down_1_plants_weather_mapped.xlsx


In [14]:
min_time,max_time

(datetime.datetime(2025, 12, 12, 15, 0, 1),
 datetime.datetime(2025, 12, 21, 11, 15, 1))

In [ ]:
import pandas as pd

def filter_datetime_excel(input_excel, output_excel, start_time, end_time):

    # read excel
    df = pd.read_excel(input_excel)

    # convert Datetime column to datetime format
    df["Date Time"] = pd.to_datetime(df["Date Time"], format="%d-%m-%Y %H:%M")

    # define time range
    start = pd.to_datetime(start_time)
    end = pd.to_datetime(end_time)

    # filter rows
    filtered_df = df[(df["Date Time"] >= start) & (df["Date Time"] <= end)]

    # save new excel
    filtered_df.to_excel(output_excel, index=False)

    print("Filtered Excel saved at:", output_excel)


input_excel = r"FAWN_2026.xlsx"
output_excel = r"front_down_1_plants_weather_mapped_2026.xlsx"

# start_time = "2025-12-12 15:00:01"
# end_time = "2026-01-18 18:45:01"

filter_datetime_excel(input_excel, output_excel, start_time=min_time, end_time=max_time)

Filtered Excel saved at: front_down_1_plants_weather_mapped_2026.xlsx


In [23]:
import os
import pandas as pd
from datetime import datetime

def map_images_to_excel(excel_path, image_folder, output_excel):

    # read excel
    df = pd.read_excel(excel_path)
    df["Date Time"] = pd.to_datetime(df["Date Time"], format="%d-%m-%Y %H:%M")

    image_data = []

    # read image timestamps
    for file in os.listdir(image_folder):
        if file.endswith(".jpg"):

            try:
                name = file.replace("citra_", "").replace(".jpg", "")
                img_time = datetime.strptime(name, "%Y-%m-%d_%H%M%S")

                image_data.append((img_time, file))

            except:
                try:
                    name = file.replace(".jpg", "")
                    img_time = datetime.strptime(name, "%Y-%m-%d_%H-%M")

                    image_data.append((img_time, file))
                except:
                    continue

    # convert to dataframe
    img_df = pd.DataFrame(image_data, columns=["img_time", "image_name"])
    img_df = img_df.sort_values("img_time")

    # map nearest timestamp
    df["image_name"] = df["Date Time"].apply(
        lambda x: img_df.iloc[(img_df["img_time"] - x).abs().argsort()[:1]]["image_name"].values[0]
    )

    # save new excel
    df.to_excel(output_excel, index=False)

    print("Mapped Excel saved:", output_excel)


excel_path = r"Front_down_1_plants_weather_mapped.xlsx"
image_folder = r"F:\strwaberry images\Front_row_with_text\Cropped_Images_down\down_1\preprocessed_images"
output_excel = r"Front_images_mapped_images_down_1.xlsx"

map_images_to_excel(excel_path, image_folder, output_excel)

Mapped Excel saved: Front_images_mapped_images_down_1.xlsx


In [28]:
import os
import pandas as pd
from datetime import datetime

def map_images_with_tolerance(excel_path, image_folder, output_excel):

    # read excel
    df = pd.read_excel(excel_path)
    df["Date Time"] = pd.to_datetime(df["Date Time"])
    df = df.sort_values("Date Time")

    image_data = []

    # read image timestamps
    for file in os.listdir(image_folder):

        if file.endswith(".jpg"):
            try:
                name = file.replace("citra_", "").replace(".jpg", "")
                img_time = datetime.strptime(name, "%Y-%m-%d_%H%M%S")
            except:
                try:
                    name = file.replace(".jpg", "")
                    img_time = datetime.strptime(name, "%Y-%m-%d_%H-%M")
                except:
                    continue

            image_data.append([img_time, file])

    img_df = pd.DataFrame(image_data, columns=["img_time", "image_name"])
    img_df = img_df.sort_values("img_time")
    print(image_data)

    # nearest merge with tolerance ±5 minutes
    mapped = pd.merge_asof(
        df,
        img_df,
        left_on="Date Time",
        right_on="img_time",
        direction="nearest",
        tolerance=pd.Timedelta("5min")
    )

    mapped.drop(columns=["img_time"], inplace=True)

    mapped.to_excel(output_excel, index=False)

    print("Saved:", output_excel)

# excel_path = r"down_plants_weather_mapped.xlsx"
# image_folder = r"back row upated with text\Cropped_Images\down_2\preprocessed_images"
# output_excel = r"images_mapped_images_down.xlsx"

excel_path = r"Front_down_1_plants_weather_mapped.xlsx"
image_folder = r"F:\strwaberry images\Front_row_with_text\Cropped_Images_down\down_1\preprocessed_images"
output_excel = r"Front_images_mapped_images_down_1.xlsx"

map_images_with_tolerance(excel_path, image_folder, output_excel)

[[datetime.datetime(2025, 12, 12, 15, 0, 1), 'citra_2025-12-12_150001.jpg'], [datetime.datetime(2025, 12, 13, 6, 0, 1), 'citra_2025-12-13_060001.jpg'], [datetime.datetime(2025, 12, 13, 6, 15, 1), 'citra_2025-12-13_061501.jpg'], [datetime.datetime(2025, 12, 13, 6, 30, 1), 'citra_2025-12-13_063001.jpg'], [datetime.datetime(2025, 12, 13, 6, 45, 1), 'citra_2025-12-13_064501.jpg'], [datetime.datetime(2025, 12, 13, 7, 0, 1), 'citra_2025-12-13_070001.jpg'], [datetime.datetime(2025, 12, 13, 7, 15, 1), 'citra_2025-12-13_071501.jpg'], [datetime.datetime(2025, 12, 13, 7, 30, 1), 'citra_2025-12-13_073001.jpg'], [datetime.datetime(2025, 12, 13, 7, 45, 1), 'citra_2025-12-13_074501.jpg'], [datetime.datetime(2025, 12, 13, 8, 0, 1), 'citra_2025-12-13_080001.jpg'], [datetime.datetime(2025, 12, 13, 8, 15, 1), 'citra_2025-12-13_081501.jpg'], [datetime.datetime(2025, 12, 13, 8, 30, 1), 'citra_2025-12-13_083001.jpg'], [datetime.datetime(2025, 12, 13, 8, 45, 1), 'citra_2025-12-13_084501.jpg'], [datetime.date

In [8]:
import os
import pandas as pd
from datetime import datetime

def map_images_with_tolerance(excel_path, image_folder, output_excel):

    # read excel
    df = pd.read_excel(excel_path)

    # let pandas auto-parse datetime
    df["Date Time"] = pd.to_datetime(df["Date Time"],format="%Y-%m-%d %H:%M:%S")

    df = df.sort_values("Date Time").reset_index(drop=True)


    image_data = []

    # read image timestamps
    for file in os.listdir(image_folder):

        if file.endswith(".jpg"):

            try:
                # citra_2025-12-13_060001.jpg
                name = file.replace("citra_", "").replace(".jpg", "")
                img_time = datetime.strptime(name, "%Y-%m-%d_%H%M%S")

                image_data.append([img_time, file])

            except:
                continue

    img_df = pd.DataFrame(image_data, columns=["img_time", "image_name"])
    img_df = img_df.sort_values("img_time").reset_index(drop=True)

    # IMPORTANT: merge_asof requires sorting
    # img_df = img_df.sort_values("img_time")
    print(img_df)

    # merge with tolerance
    mapped = pd.merge_asof(
        df,
        img_df,
        left_on="Date Time",
        right_on="img_time",
        direction="nearest",
        tolerance=pd.Timedelta("2min")
    )
    # print(mapped)
    return img_df,mapped,df
    # mapped.drop(columns=["img_time"], inplace=True)

    # mapped.to_excel(output_excel, index=False)

    print("Saved:", output_excel)

excel_path = r"Front_down_1_plants_weather_mapped.xlsx"
image_folder = r"F:\strwaberry images\Front_row_with_text\Cropped_Images_down\down_1\preprocessed_images"
output_excel = r"Front_images_mapped_images_down_11.xlsx"
a,b,c=map_images_with_tolerance(excel_path, image_folder, output_excel)

               img_time                   image_name
0   2025-12-12 15:00:01  citra_2025-12-12_150001.jpg
1   2025-12-13 06:00:01  citra_2025-12-13_060001.jpg
2   2025-12-13 06:15:01  citra_2025-12-13_061501.jpg
3   2025-12-13 06:30:01  citra_2025-12-13_063001.jpg
4   2025-12-13 06:45:01  citra_2025-12-13_064501.jpg
..                  ...                          ...
316 2025-12-21 10:15:01  citra_2025-12-21_101501.jpg
317 2025-12-21 10:30:01  citra_2025-12-21_103001.jpg
318 2025-12-21 10:45:01  citra_2025-12-21_104501.jpg
319 2025-12-21 11:00:01  citra_2025-12-21_110001.jpg
320 2025-12-21 11:15:01  citra_2025-12-21_111501.jpg

[321 rows x 2 columns]


In [11]:
b.to_excel('mapped_excel_front_down_1.xlsx', index=False)

In [10]:
b

,Station ID,Date Time,Soil Temp (C),Temp @ 60cm (C),Temp @ 2m (C),Temp @ 10m (C),Relative Humidity (%),Dew Point Temp (C),Rainfall Amount (in),Wind Speed (mph),Wind Direction (deg),Solar Radiation (w/m2),img_time,image_name
0,260,2025-12-12 15:15:00,14.84,23.08,22.61,21.34,29.56,3.96,0.0,4.79,236.60,379.7,NaT,NaN
1,260,2025-12-12 15:30:00,15.00,23.10,22.18,21.14,30.02,3.80,0.0,5.77,216.70,340.9,NaT,NaN
2,260,2025-12-12 15:45:00,15.15,22.48,22.34,21.30,29.40,3.65,0.0,4.81,238.00,299.2,NaT,NaN
3,260,2025-12-12 16:00:00,15.28,22.42,22.20,21.47,29.07,3.37,0.0,3.94,253.10,257.0,NaT,NaN
4,260,2025-12-12 16:15:00,15.40,22.29,22.19,21.36,30.09,3.85,0.0,3.90,225.70,213.7,NaT,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
844,260,2025-12-21 10:15:00,12.44,14.03,15.15,13.73,86.50,12.91,0.0,1.82,41.25,460.3,2025-12-21 10:15:01,citra_2025-12-21_101501.jpg
845,260,2025-12-21 10:30:00,12.46,16.09,16.92,15.31,68.58,11.11,0.0,3.51,31.14,486.9,2025-12-21 10:30:01,citra_2025-12-21_103001.jpg
846,260,2025-12-21 10:45:00,12.49,17.40,17.72,16.29,65.73,11.23,0.0,4.46,35.11,508.2,2025-12-21 10:45:01,citra_2025-12-21_104501.jpg
847,260,2025-12-21 11:00:00,12.54,18.48,18.97,17.27,62.68,11.70,0.0,3.78,44.05,532.5,2025-12-21 11:00:01,citra_2025-12-21_110001.jpg


In [12]:
import pandas as pd
import numpy as np

excel_path = r"F:\strwaberry images\Front_images_mapped_images_down_1.xlsx"

df = pd.read_excel(excel_path)

# convert datetime
df["Date Time"] = pd.to_datetime(df["Date Time"])

# extract date
df["Date"] = df["Date Time"].dt.date

# temperature column
temp_col = "Temp @ 2m (C)"

# calculate daily Tmin and Tmax
daily_temp = df.groupby("Date")[temp_col].agg(
    Tmin="min",
    Tmax="max"
).reset_index()

# parameters
Tbase = 7
Tmax_limit = 30

# apply rules
daily_temp["Tmax"] = daily_temp["Tmax"].clip(upper=Tmax_limit)
daily_temp["Tmin"] = daily_temp["Tmin"].clip(lower=Tbase)

# calculate GDD
daily_temp["GDD"] = ((daily_temp["Tmax"] + daily_temp["Tmin"]) / 2) - Tbase

# rule: negative GDD = 0
daily_temp["GDD"] = daily_temp["GDD"].clip(lower=0)

# cumulative GDD
daily_temp["Cumulative_GDD"] = daily_temp["GDD"].cumsum()

print(daily_temp.head())

# save
daily_temp.to_excel(r"F:\Front_down1_strawberry_gdd.xlsx", index=False)

         Date  Tmin   Tmax    GDD  Cumulative_GDD
0  2025-12-12  7.00  22.61  7.805           7.805
1  2025-12-13  7.00  25.09  9.045          16.850
2  2025-12-14  7.26  24.89  9.075          25.925
3  2025-12-15  7.00  13.32  3.160          29.085
4  2025-12-16  7.00  20.02  6.510          35.595


In [13]:
file = r"Front_images_mapped_images_down_1.xlsx"
df = pd.read_excel(file)

# convert datetime
df["Date Time"] = pd.to_datetime(df["Date Time"])

# extract date
df["date"] = df["Date Time"].dt.date

temp_col = "Temp @ 2m (C)"

# calculate daily Tmax and Tmin
daily = df.groupby("date")[temp_col].agg(["max","min"]).reset_index()
daily.columns = ["date","Tmax","Tmin"]

# GDD parameters
Tbase = 7
Tmax_cap = 30

# apply rules
daily["Tmax"] = daily["Tmax"].clip(upper=Tmax_cap)
daily["Tmin"] = daily["Tmin"].clip(lower=Tbase)

daily["GDD"] = ((daily["Tmax"] + daily["Tmin"]) / 2) - Tbase
daily["GDD"] = daily["GDD"].clip(lower=0)

# merge back into main dataset
df_final = df.merge(daily, on="date", how="left")
df_final

,Station ID,Date Time,Soil Temp (C),Temp @ 60cm (C),Temp @ 2m (C),Temp @ 10m (C),Relative Humidity (%),Dew Point Temp (C),Rainfall Amount (in),Wind Speed (mph),Wind Direction (deg),Solar Radiation (w/m2),image_name,date,Tmax,Tmin,GDD
0,260,2025-12-12 15:15:00,14.84,23.08,22.61,21.34,29.56,3.96,0.0,4.79,236.60,379.7,NaN,2025-12-12,22.61,7.0,7.805
1,260,2025-12-12 15:30:00,15.00,23.10,22.18,21.14,30.02,3.80,0.0,5.77,216.70,340.9,NaN,2025-12-12,22.61,7.0,7.805
2,260,2025-12-12 15:45:00,15.15,22.48,22.34,21.30,29.40,3.65,0.0,4.81,238.00,299.2,NaN,2025-12-12,22.61,7.0,7.805
3,260,2025-12-12 16:00:00,15.28,22.42,22.20,21.47,29.07,3.37,0.0,3.94,253.10,257.0,NaN,2025-12-12,22.61,7.0,7.805
4,260,2025-12-12 16:15:00,15.40,22.29,22.19,21.36,30.09,3.85,0.0,3.90,225.70,213.7,NaN,2025-12-12,22.61,7.0,7.805
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
844,260,2025-12-21 10:15:00,12.44,14.03,15.15,13.73,86.50,12.91,0.0,1.82,41.25,460.3,citra_2025-12-21_101501.jpg,2025-12-21,19.24,7.0,6.120
845,260,2025-12-21 10:30:00,12.46,16.09,16.92,15.31,68.58,11.11,0.0,3.51,31.14,486.9,citra_2025-12-21_103001.jpg,2025-12-21,19.24,7.0,6.120
846,260,2025-12-21 10:45:00,12.49,17.40,17.72,16.29,65.73,11.23,0.0,4.46,35.11,508.2,citra_2025-12-21_104501.jpg,2025-12-21,19.24,7.0,6.120
847,260,2025-12-21 11:00:00,12.54,18.48,18.97,17.27,62.68,11.70,0.0,3.78,44.05,532.5,citra_2025-12-21_110001.jpg,2025-12-21,19.24,7.0,6.120


In [16]:
import cv2
import numpy as np
import pandas as pd
from pathlib import Path

# load your main dataset
# df_final = pd.read_excel(r"F:\dataset_with_gdd.xlsx")

folder = r"F:\strwaberry images\Front_row_with_text\Cropped_Images_down\down_1\preprocessed_images"

green_dict = {}

for path in Path(folder).glob("*.jpg"):

    img = cv2.imread(str(path))
    if img is None:
        continue

    hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

    lower_green = np.array([35,40,40])
    upper_green = np.array([90,255,255])

    mask = cv2.inRange(hsv, lower_green, upper_green)

    kernel = np.ones((5,5),np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)

    green_pixels = cv2.countNonZero(mask)

    green_dict[path.name] = green_pixels


# add green_pixels column using image_name
df_final["green_pixels"] = df_final["image_name"].map(green_dict)
df_final

,Station ID,Date Time,Soil Temp (C),Temp @ 60cm (C),Temp @ 2m (C),Temp @ 10m (C),Relative Humidity (%),Dew Point Temp (C),Rainfall Amount (in),Wind Speed (mph),Wind Direction (deg),Solar Radiation (w/m2),image_name,date,Tmax,Tmin,GDD,green_pixels
0,260,2025-12-12 15:15:00,14.84,23.08,22.61,21.34,29.56,3.96,0.0,4.79,236.60,379.7,NaN,2025-12-12,22.61,7.0,7.805,NaN
1,260,2025-12-12 15:30:00,15.00,23.10,22.18,21.14,30.02,3.80,0.0,5.77,216.70,340.9,NaN,2025-12-12,22.61,7.0,7.805,NaN
2,260,2025-12-12 15:45:00,15.15,22.48,22.34,21.30,29.40,3.65,0.0,4.81,238.00,299.2,NaN,2025-12-12,22.61,7.0,7.805,NaN
3,260,2025-12-12 16:00:00,15.28,22.42,22.20,21.47,29.07,3.37,0.0,3.94,253.10,257.0,NaN,2025-12-12,22.61,7.0,7.805,NaN
4,260,2025-12-12 16:15:00,15.40,22.29,22.19,21.36,30.09,3.85,0.0,3.90,225.70,213.7,NaN,2025-12-12,22.61,7.0,7.805,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
844,260,2025-12-21 10:15:00,12.44,14.03,15.15,13.73,86.50,12.91,0.0,1.82,41.25,460.3,citra_2025-12-21_101501.jpg,2025-12-21,19.24,7.0,6.120,132909.0
845,260,2025-12-21 10:30:00,12.46,16.09,16.92,15.31,68.58,11.11,0.0,3.51,31.14,486.9,citra_2025-12-21_103001.jpg,2025-12-21,19.24,7.0,6.120,145134.0
846,260,2025-12-21 10:45:00,12.49,17.40,17.72,16.29,65.73,11.23,0.0,4.46,35.11,508.2,citra_2025-12-21_104501.jpg,2025-12-21,19.24,7.0,6.120,138203.0
847,260,2025-12-21 11:00:00,12.54,18.48,18.97,17.27,62.68,11.70,0.0,3.78,44.05,532.5,citra_2025-12-21_110001.jpg,2025-12-21,19.24,7.0,6.120,129498.0


In [18]:
from plotly import express as px
df=df_final
df["Date Time"] = pd.to_datetime(df["Date Time"])
df["date"] = df["Date Time"].dt.date

# df = df.dropna(subset=["green_pixels"])
daily = df_final.groupby("date").agg({
    "green_pixels":"mean",
    "GDD":"first"
}).reset_index()

# plot
fig = px.scatter(
    daily,
    x="date",
    y="green_pixels",
    color="GDD",
    title="Front row donwn 1 plant Strawberry Growth Over Time (Green Pixels) with GDD",
    labels={
        "date":"Date",
        "green_pixels":"Average Green Pixel Area",
        "GDD":"Daily GDD"
    }
)

fig.update_traces(marker=dict(size=9))

fig.show()

In [19]:
df.to_excel('Front_Images_mapped_down1_gdd.xlsx')

In [22]:
daily = df.groupby("date").agg({
    "Solar Radiation (w/m2)": "sum",
    "green_pixels": "sum"
}).reset_index()

In [23]:
import plotly.express as px

fig = px.scatter(
    daily,
    x="Solar Radiation (w/m2)",
    y="green_pixels",
    trendline="ols",
    title="Daily Solar Radiation vs Plant Size"
)

fig.show()

In [2]:
from ultralytics import YOLO
import pandas as pd
import os

# Load model
model = YOLO("best.pt")

image_folder = r"F:\strwaberry images\Front_row_with_text\Cropped_Images_down\down_1\preprocessed_images"
results_list = []
failed_images = []

for img in os.listdir(image_folder):

    img_path = os.path.join(image_folder, img)

    try:
        # Run inference
        results = model(img_path)

        class_counts = {}
        total = 0

        for r in results:
            if r.boxes is None:
                continue

            boxes = r.boxes.cls.tolist()

            for cls in boxes:
                class_name = model.names[int(cls)]

                class_counts[class_name] = class_counts.get(class_name, 0) + 1
                total += 1

        row = {
            "image_name": img,
            "total_objects": total
        }

        row.update(class_counts)
        results_list.append(row)

    except Exception as e:
        print(f"Error processing {img}: {e}")
        failed_images.append(img)
        continue

# Convert to dataframe
df = pd.DataFrame(results_list).fillna(0)

# Save detection results
df.to_excel("yolo_results_front.xlsx", index=False)

# Save failed images list
if failed_images:
    failed_df = pd.DataFrame({"failed_images": failed_images})
    failed_df.to_excel("failed_images_front.xlsx", index=False)

print("Processing complete.")
print(f"Successful images: {len(results_list)}")
print(f"Failed images: {len(failed_images)}")


image 1/1 F:\strwaberry images\Front_row_with_text\Cropped_Images_down\down_1\preprocessed_images\citra_2025-12-12_150001.jpg: 480x640 2 Gs, 1 W, 1 P, 2 Rs, 532.9ms
Speed: 4.0ms preprocess, 532.9ms inference, 1.6ms postprocess per image at shape (1, 3, 480, 640)

image 1/1 F:\strwaberry images\Front_row_with_text\Cropped_Images_down\down_1\preprocessed_images\citra_2025-12-13_060001.jpg: 480x640 1 R, 458.1ms
Speed: 4.1ms preprocess, 458.1ms inference, 1.3ms postprocess per image at shape (1, 3, 480, 640)

image 1/1 F:\strwaberry images\Front_row_with_text\Cropped_Images_down\down_1\preprocessed_images\citra_2025-12-13_061501.jpg: 480x640 2 Rs, 508.2ms
Speed: 3.9ms preprocess, 508.2ms inference, 1.4ms postprocess per image at shape (1, 3, 480, 640)

image 1/1 F:\strwaberry images\Front_row_with_text\Cropped_Images_down\down_1\preprocessed_images\citra_2025-12-13_063001.jpg: 480x640 1 P, 1 R, 514.7ms
Speed: 3.6ms preprocess, 514.7ms inference, 2.8ms postprocess per image at shape (1, 3,